# 07b - Final fine-tuned evaluation in the frozen inference stack

This notebook produces the final fine-tuned numbers for the selected adapters. It runs in a **fresh runtime** with the same inference stack that produced the base zero-shot and few-shot results:


In [ ]:
!pip install -q transformers==5.20.1 accelerate bitsandbytes==0.50.2 peft huggingface_hub

import transformers, torch, bitsandbytes, peft
print("transformers ", transformers.__version__)
print("torch        ", torch.__version__)
print("bitsandbytes ", bitsandbytes.__version__)
print("peft         ", peft.__version__)
print("GPU          ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert transformers.__version__ == "5.20.1", transformers.__version__
assert bitsandbytes.__version__ == "0.50.2", bitsandbytes.__version__
assert torch.__version__.startswith("2.11.0"), torch.__version__
assert torch.cuda.is_available(), "Switch the Colab runtime to GPU (T4) before running this notebook."


transformers  5.20.1
torch         2.11.0+cu130
bitsandbytes  0.50.2
peft          0.21.2
GPU           Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, gc
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch, transformers, bitsandbytes, peft
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login

try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok:
        login(token=_tok)
        print("HF login: token from Colab userdata")
    else:
        print("HF login: no HF_TOKEN in Colab userdata; relying on the local cache")
except Exception as e:
    print("HF login skipped:", type(e).__name__)

BASE = "/content/drive/MyDrive/cxr_eval_data"
FINAL = os.path.join(BASE, "final_eval")
os.makedirs(FINAL, exist_ok=True)
MODEL = "meta-llama/Llama-3.2-3B-Instruct"
MAX_NEW = {"direct": 200, "structure": 300}
RUN_PIPELINES = ["direct", "structure"]

need = ["prompt_templates.json", "val_set.csv", "test_set.csv",
        os.path.join("lora_search", "selected_configs.json")]
missing = [n for n in need if not os.path.exists(os.path.join(BASE, n))]
print(f"Found {len(need) - len(missing)}/{len(need)} required files in {BASE}")
if missing:
    for n in missing:
        print("  MISSING", n)
    raise FileNotFoundError("Run 05, 06 and 07a first, then rerun this cell.")

with open(os.path.join(BASE, "lora_search", "selected_configs.json"), encoding="utf-8") as f:
    SELECTED = json.load(f)
assert set(SELECTED) == {"direct", "structure"}, set(SELECTED)
for pipe in RUN_PIPELINES:
    ad = SELECTED[pipe]["adapter_dir"]
    assert os.path.isdir(ad), f"Adapter directory missing: {ad}"
    assert os.path.exists(os.path.join(ad, "adapter_config.json")), ad
print("Selected configurations:", {p: SELECTED[p]["config"] for p in SELECTED})

with open(os.path.join(BASE, "prompt_templates.json"), encoding="utf-8") as f:
    TPL = json.load(f)
assert set(TPL) == {"direct", "structure"}

val = pd.read_csv(os.path.join(BASE, "val_set.csv"))
assert len(val) == 395, len(val)
assert val["id"].is_unique
print(f"val={len(val)} (test is loaded only inside the gated test cell)")

PC = ['No Finding','Enlarged Cardiomediastinum','Cardiomegaly','Lung Lesion','Lung Opacity','Edema',
      'Consolidation','Pneumonia','Atelectasis','Pneumothorax','Pleural Effusion','Pleural Other',
      'Fracture','Support Devices']
CLIN = [p for p in PC if p != 'No Finding']

PATH_COLS = {
    "lungs_and_airways.lung_opacity": "Lung Opacity", "lungs_and_airways.consolidation": "Consolidation",
    "lungs_and_airways.edema": "Edema", "lungs_and_airways.atelectasis": "Atelectasis",
    "lungs_and_airways.lung_lesion": "Lung Lesion", "lungs_and_airways.pneumonia": "Pneumonia",
    "pleura.pleural_effusion": "Pleural Effusion", "pleura.pneumothorax": "Pneumothorax",
    "pleura.pleural_other": "Pleural Other", "cardiovascular.cardiomegaly": "Cardiomegaly",
    "cardiovascular.enlarged_cardiomediastinum": "Enlarged Cardiomediastinum",
    "bones_and_chest_wall.fracture": "Fracture", "devices.support_devices": "Support Devices",
    "general.no_finding": "No Finding",
}
VMAP = {"present": "1", "absent": "0", "uncertain": "-1", "not_mentioned": "NaN", "nan": "NaN"}

def parse_direct(raw):
    parsed = {}
    for line in raw.strip().split("\n"):
        line = line.strip()
        if "=" in line:
            parts = line.split("=")
            if len(parts) == 2:
                name, value = parts[0].strip(), parts[1].strip()
                if name in PC and value in ("1", "0", "-1", "NaN", "nan"):
                    parsed[name] = value
    return [parsed.get(p, "NaN") for p in PC], len(parsed)

def parse_struct(raw):
    s, e = raw.find("{"), raw.rfind("}")
    if s == -1 or e <= s:
        return ["NaN"] * 14, 0
    try:
        data = json.loads(raw[s:e + 1])
    except Exception:
        return ["NaN"] * 14, 0
    out = {}
    for path, p in PATH_COLS.items():
        cur = data
        ok = True
        for k in path.split("."):
            if isinstance(cur, dict) and k in cur:
                cur = cur[k]
            else:
                ok = False
                break
        if ok and isinstance(cur, str) and cur.strip().lower() in VMAP:
            out[p] = VMAP[cur.strip().lower()]
    return [out.get(p, "NaN") for p in PC], len(out)

def parse_any(pipe, raw):
    return parse_direct(raw) if pipe == "direct" else parse_struct(raw)

def truth_arrays(df):
    out = {}
    for p in PC:
        t = df[p].map(lambda x: np.nan if pd.isna(x) else float(x)).to_numpy()
        out[p] = np.where(t == -1, np.nan, np.where(np.isnan(t), 0.0, t))
    return out

def macro_scores(values, truth):
    per = {}
    for j, p in enumerate(PC):
        t = truth[p]
        pr = np.array([1.0 if v[j] == "1" else 0.0 for v in values])
        tp = int(((t == 1) & (pr == 1)).sum())
        fp = int(((t == 0) & (pr == 1)).sum())
        fn = int(((t == 1) & (pr == 0)).sum())
        d = 2 * tp + fp + fn
        per[p] = 2 * tp / d if d else 0.0
    return {"macro13": float(np.mean([per[p] for p in CLIN])),
            "macro14": float(np.mean(list(per.values()))),
            "NoFinding": per["No Finding"]}

def adapter_info(pipe):
    p = os.path.join(SELECTED[pipe]["adapter_dir"], "adapter_config.json")
    with open(p, encoding="utf-8") as f:
        c = json.load(f)
    keys = ("peft_type", "r", "lora_alpha", "lora_dropout", "target_modules", "bias",
            "base_model_name_or_path")
    return {k: c.get(k) for k in keys}

def write_record(extra=None):
    rec = {
        "recorded_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "gpu": torch.cuda.get_device_name(0),
        "versions": {"transformers": transformers.__version__, "torch": torch.__version__,
                     "bitsandbytes": bitsandbytes.__version__, "peft": peft.__version__},
        "model": MODEL,
        "quantization": {"load_in_4bit": True, "bnb_4bit_quant_type": "nf4",
                         "bnb_4bit_compute_dtype": "float16"},
        "decoding": {"do_sample": False, "repetition_penalty": 1.1,
                     "max_new_tokens": MAX_NEW, "batch_size": 1},
        "selected": SELECTED,
        "adapters": {p: adapter_info(p) for p in RUN_PIPELINES},
        "files": {},
    }
    for name in ("val_sanity.csv", "test_metrics.csv"):
        p = os.path.join(FINAL, name)
        if os.path.exists(p):
            rec["files"][name] = pd.read_csv(p).to_dict(orient="records")
    if extra:
        rec.update(extra)
    p = os.path.join(FINAL, "environment_07b.json")
    with open(p, "w", encoding="utf-8") as f:
        json.dump(rec, f, indent=1, ensure_ascii=False)
    print("Wrote", p)
    return rec

print("Scorer ready: uncertain truth excluded, blank -> absent, only present positive, all reports scored.")


Mounted at /content/drive
HF login: token from Colab userdata
Found 4/4 required files in /content/drive/MyDrive/cxr_eval_data
Selected configurations: {'direct': 'direct_r16_a32_lr0.0002_ep2_s42', 'structure': 'structure_r32_a32_lr0.0002_ep2_s42'}
val=395 (test is loaded only inside the gated test cell)
Scorer ready: uncertain truth excluded, blank -> absent, only present positive, all reports scored.


## Model loading and generation functions

In [ ]:
@torch.no_grad()
def generate_one(model, tok, pipe, report):
    prompt = TPL[pipe]["prefix"] + str(report) + TPL[pipe]["suffix"]
    formatted = tok.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    enc = tok(formatted, return_tensors="pt").to(model.device)
    n_prompt = enc["input_ids"].shape[1]
    if n_prompt > 2048 - MAX_NEW[pipe]:
        raise RuntimeError(f"Prompt too long for the frozen max length: {n_prompt} tokens")
    out = model.generate(**enc, max_new_tokens=MAX_NEW[pipe],
                         do_sample=False, repetition_penalty=1.1)
    gen = out[0][n_prompt:]
    return tok.decode(gen, skip_special_tokens=True)

def load_pipe(pipe):
    adapter_dir = SELECTED[pipe]["adapter_dir"]
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tok = AutoTokenizer.from_pretrained(MODEL)
    base = AutoModelForCausalLM.from_pretrained(
        MODEL, quantization_config=bnb, device_map="auto")
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.eval()
    info = adapter_info(pipe)
    print(f"Loaded {pipe}: {SELECTED[pipe]['config']}")
    print("  adapter:", {k: info[k] for k in ("peft_type", "r", "lora_alpha")})
    return model, tok

def load_jsonl(path):
    rows = []
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return rows

def rewrite_jsonl(path, rows):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    os.replace(tmp, path)

def run_split(pipe, split, df, model, tok):
    outdir = os.path.join(FINAL, pipe)
    os.makedirs(outdir, exist_ok=True)
    path = os.path.join(outdir, f"{split}_raw.jsonl")
    rows = load_jsonl(path)
    rewrite_jsonl(path, rows)
    by_id = {int(r["id"]): r for r in rows}
    assert len(by_id) == len(rows), "duplicate ids in " + path
    todo = [i for i in range(len(df)) if int(df.iloc[i]["id"]) not in by_id]
    print(f"\n=== {pipe} / {split}: {len(by_id)}/{len(df)} reports already generated; {len(todo)} to go ===")
    t0 = time.time()
    with open(path, "a", encoding="utf-8") as f:
        for n, i in enumerate(todo, 1):
            s = df.iloc[i]
            raw = generate_one(model, tok, pipe, s["full_report"])
            pred, n_found = parse_any(pipe, raw)
            truth = [None if pd.isna(s[p]) else float(s[p]) for p in PC]
            rec = {"id": int(s["id"]), "raw": raw, "pred": pred,
                   "truth": truth, "n_found": n_found}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            f.flush()
            by_id[rec["id"]] = rec
            if n % 10 == 0 or n == len(todo):
                print(f"  generated {n}/{len(todo)}", end="\r")
    print()

    missing = [int(s["id"]) for _, s in df.iterrows() if int(s["id"]) not in by_id]
    assert not missing, f"{len(missing)} reports missing from {path}"

    out_rows = []
    for _, s in df.iterrows():
        r = by_id[int(s["id"])]
        row = {"id": int(s["id"]), "raw": r["raw"]}
        for p in PC:
            row[f"truth_{p}"] = s[p]
        for j, p in enumerate(PC):
            row[f"pred_{p}"] = r["pred"][j]
        row["n_found"] = r["n_found"]
        out_rows.append(row)
    res_csv = os.path.join(outdir, f"{split}_predictions.csv")
    pd.DataFrame(out_rows).to_csv(res_csv, index=False)

    values = [by_id[int(s["id"])]["pred"] for _, s in df.iterrows()]
    ms = macro_scores(values, truth_arrays(df))
    found = np.array([by_id[int(s["id"])]["n_found"] for _, s in df.iterrows()])
    ms.update({
        "pipeline": pipe, "split": split, "n": len(df),
        "valid14": float(np.mean(found == 14)),
        "usable_any": float(np.mean(found > 0)),
        "generated_now": len(todo),
        "already_done": len(df) - len(todo),
        "minutes": (time.time() - t0) / 60,
        "results_csv": res_csv, "raw_jsonl": path,
        "recorded_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    })
    print(f"  macro13={ms['macro13']:.3f}  macro14={ms['macro14']:.3f}  "
          f"NoFind={ms['NoFinding']:.3f}  valid14={ms['valid14']*100:.1f}%  "
          f"usable={ms['usable_any']*100:.1f}%  ({ms['minutes']:.1f} min)")
    return ms


## Validation sanity check

In [ ]:
VAL_ROWS = []
for pipe in RUN_PIPELINES:
    model, tok = load_pipe(pipe)
    m = run_split(pipe, "val", val, model, tok)
    m["search_val_macro13"] = float(SELECTED[pipe]["val_macro13"])
    m["delta_vs_search"] = m["macro13"] - m["search_val_macro13"]
    VAL_ROWS.append(m)
    del model, tok
    gc.collect()
    torch.cuda.empty_cache()

val_df = pd.DataFrame(VAL_ROWS)
val_df.to_csv(os.path.join(FINAL, "val_sanity.csv"), index=False)
print("\nValidation: frozen stack vs search-time score")
print(val_df[["pipeline", "n", "macro13", "macro14", "NoFinding", "valid14",
              "search_val_macro13", "delta_vs_search", "generated_now", "minutes"]].to_string(index=False))
print("\nA small delta is expected because the adapter is attached to a freshly quantised base.")
print("Review this table before setting RUN_TEST = True in the next cell.")
write_record()


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (3 outputs left out)

Loaded direct: direct_r16_a32_lr0.0002_ep2_s42
  adapter: {'peft_type': 'LORA', 'r': 16, 'lora_alpha': 32}

=== direct / val: 395/395 reports already generated; 0 to go ===

  macro13=0.843  macro14=0.852  NoFind=0.964  valid14=100.0%  usable=100.0%  (0.0 min)


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loaded structure: structure_r32_a32_lr0.0002_ep2_s42
  adapter: {'peft_type': 'LORA', 'r': 32, 'lora_alpha': 32}

=== structure / val: 395/395 reports already generated; 0 to go ===

  macro13=0.880  macro14=0.885  NoFind=0.953  valid14=99.7%  usable=100.0%  (0.0 min)

Validation: frozen stack vs search-time score
 pipeline   n  macro13  macro14  NoFinding  valid14  search_val_macro13  delta_vs_search  generated_now  minutes
   direct 395 0.843246 0.851900   0.964401 1.000000            0.844913        -0.001667              0 0.008370
structure 395 0.880257 0.885472   0.953271 0.997468            0.875457         0.004800              0 0.007737

A small delta is expected because the adapter is attached to a freshly quantised base.
Review this table before setting RUN_TEST = True in the next cell.
Wrote /content/drive/MyDrive/cxr_eval_data/final_eval/environment_07b.json


{'recorded_utc': '2026-10-25T10:26:53+00:00',
 'gpu': 'Tesla T4',
 'versions': {'transformers': '5.20.1',
  'torch': '2.11.0+cu130',
  'bitsandbytes': '0.50.2',
  'peft': '0.21.2'},
 'model': 'meta-llama/Llama-3.2-3B-Instruct',
 'quantization': {'load_in_4bit': True,
  'bnb_4bit_quant_type': 'nf4',
  'bnb_4bit_compute_dtype': 'float16'},
 'decoding': {'do_sample': False,
  'repetition_penalty': 1.1,
  'max_new_tokens': {'direct': 200, 'structure': 300},
  'batch_size': 1},
 'selected': {'direct': {'config': 'direct_r16_a32_lr0.0002_ep2_s42',
   'r': 16,
   'alpha': 32,
   'lr': 0.0002,
   'epochs': 2,
   'seed': 42,
   'val_macro13': 0.8449131927140747,
   'val_macro14': 0.8534314918850333,
   'val_NoFinding': 0.9641693811074918,
   'val_valid14': 1.0,
   'adapter_dir': '/content/drive/MyDrive/cxr_eval_data/lora_search/direct/direct_r16_a32_lr0.0002_ep2_s42/adapter',
   'selected_utc': '2026-10-24T13:52:45+00:00',
   'selection_rule': 'highest val macro13; ties: lower rank, fewer epoch

In [ ]:
# Test evaluation 
RUN_TEST = True

if not RUN_TEST:
    print("Test not run. Review the validation table, then set RUN_TEST = True and rerun this cell.")
else:
    test = pd.read_csv(os.path.join(BASE, "test_set.csv"))
    assert len(test) == 792, len(test)
    assert test["id"].is_unique
    print(f"test={len(test)}")

    TEST_ROWS = []
    for pipe in RUN_PIPELINES:
        model, tok = load_pipe(pipe)
        m = run_split(pipe, "test", test, model, tok)
        TEST_ROWS.append(m)
        del model, tok
        gc.collect()
        torch.cuda.empty_cache()

    test_df = pd.DataFrame(TEST_ROWS)
    test_df.to_csv(os.path.join(FINAL, "test_metrics.csv"), index=False)
    print("\nFinal test results")
    print(test_df[["pipeline", "n", "macro13", "macro14", "NoFinding", "valid14",
                   "usable_any", "generated_now", "minutes"]].to_string(index=False))
    print("\nAll 792 test reports scored per pipeline; failed outputs counted as absent.")
    write_record()


test=792


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loaded direct: direct_r16_a32_lr0.0002_ep2_s42
  adapter: {'peft_type': 'LORA', 'r': 16, 'lora_alpha': 32}

=== direct / test: 792/792 reports already generated; 0 to go ===

  macro13=0.846  macro14=0.853  NoFind=0.941  valid14=99.9%  usable=100.0%  (0.0 min)


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loaded structure: structure_r32_a32_lr0.0002_ep2_s42
  adapter: {'peft_type': 'LORA', 'r': 32, 'lora_alpha': 32}

=== structure / test: 632/792 reports already generated; 160 to go ===


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  generated 160/160
  macro13=0.883  macro14=0.886  NoFind=0.937  valid14=100.0%  usable=100.0%  (40.7 min)

Final test results
 pipeline   n  macro13  macro14  NoFinding  valid14  usable_any  generated_now   minutes
   direct 792 0.846091 0.852882   0.941176 0.998737         1.0              0  0.010637
structure 792 0.882535 0.886434   0.937126 1.000000         1.0            160 40.700948

All 792 test reports scored per pipeline; failed outputs counted as absent.
Wrote /content/drive/MyDrive/cxr_eval_data/final_eval/environment_07b.json


## Final artifacts

In [6]:
# ===== Final artifacts and session record =====
for name in ("val_sanity.csv", "test_metrics.csv"):
    p = os.path.join(FINAL, name)
    if os.path.exists(p):
        print("\n" + name)
        print(pd.read_csv(p).to_string(index=False))

rec = write_record()
print("\nRaw outputs and predictions:")
for pipe in RUN_PIPELINES:
    for split in ("val", "test"):
        for kind in ("raw.jsonl", "predictions.csv"):
            p = os.path.join(FINAL, pipe, f"{split}_{kind}")
            print(f"  {p}  {'OK' if os.path.exists(p) else 'missing'}")

print("\nSend back: final_eval/val_sanity.csv, final_eval/test_metrics.csv,")
print("final_eval/<pipeline>/test_predictions.csv, and final_eval/environment_07b.json.")
print("The test set has now been evaluated once; do not regenerate it.")



val_sanity.csv
 macro13  macro14  NoFinding  pipeline split   n  valid14  usable_any  generated_now  already_done  minutes                                                                   results_csv                                                               raw_jsonl              recorded_utc  search_val_macro13  delta_vs_search
0.843246 0.851900   0.964401    direct   val 395 1.000000         1.0              0           395 0.008370    /content/drive/MyDrive/cxr_eval_data/final_eval/direct/val_predictions.csv    /content/drive/MyDrive/cxr_eval_data/final_eval/direct/val_raw.jsonl 2026-10-25T10:26:19+00:00            0.844913        -0.001667
0.880257 0.885472   0.953271 structure   val 395 0.997468         1.0              0           395 0.007737 /content/drive/MyDrive/cxr_eval_data/final_eval/structure/val_predictions.csv /content/drive/MyDrive/cxr_eval_data/final_eval/structure/val_raw.jsonl 2026-10-25T10:26:52+00:00            0.875457         0.004800

test_metrics.csv
 ma